<a href="https://colab.research.google.com/github/fufaev-dev/cookies/blob/main/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Установка ComfyUI
!git clone -q https://github.com/comfyanonymous/ComfyUI
%pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
%pip install -q -r ./ComfyUI/requirements.txt

# Базовые полезные узлы
!git clone -q https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/comfyui-manager
!git clone -q https://github.com/rgthree/rgthree-comfy.git /content/ComfyUI/custom_nodes/rgthree-comfy
!git clone -q https://github.com/yolain/ComfyUI-Easy-Use.git /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use

# Узлы для Krita
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!git clone https://github.com/Acly/comfyui-tooling-nodes /content/ComfyUI/custom_nodes/comfyui-tooling-nodes
!git clone https://github.com/Acly/comfyui-inpaint-nodes /content/ComfyUI/custom_nodes/comfyui-inpaint-nodes
!git clone https://github.com/ssitu/ComfyUI_UltimateSDUpscale /content/ComfyUI/custom_nodes/ComfyUI_UltimateSDUpscale

# Устанавливаем зависимости и решаем конфликт
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/requirements.txt
!pip install -q insightface onnxruntime onnxruntime-gpu argostranslate
!pip uninstall -y albumentations
!pip install -q albucore==0.3.10 albumentationsx==2.6.6
!pip cache purge

print("✅ Шаг 1: База и узлы установлены.")

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
Cloning into '/content/ComfyUI/custom_nodes/comfyui_controlnet_aux'...
remote: Enumerating objects: 5857, done.
remote: Counting objects: 100% (2121/2121), done.
remote: Compressing objects: 100% (507/507), done.
remote: Total 5857 (delta 1797), reused 1623 (delta 1614), pack-reused 3736 (from 2)
Receiving objects: 100% (5857/5857), 42.97 MiB | 30.41 MiB/s, done.
Resolving deltas: 100% (3197/3197), done.
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'...
remote: Enumerating objects: 813, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 813 (delta 404), reused 297 (delta 297), pack-reused 384 (from 2)
Receiving objects: 100% (813/813), 4.73 MiB | 28.49 MiB/s, done.
Resolving deltas: 100% (551/551), done.
Cloning into '/content/ComfyUI/custom_nodes/comfyui-tooling-nodes'...
remote: Enumerating objects: 588, done.
rem

In [2]:
# Модели для Krita (Апскейл и Инпейнт)
!wget -q --show-progress https://huggingface.co/gemasai/4x_NMKD-Superscale-SP_178000_G/resolve/main/4x_NMKD-Superscale-SP_178000_G.pth -P /content/ComfyUI/models/upscale_models/
!wget -q --show-progress https://huggingface.co/Acly/MAT/resolve/main/MAT_Places512_G_fp16.safetensors -P /content/ComfyUI/models/inpaint/
!wget -q --show-progress https://huggingface.co/Acly/Omni-SR/resolve/main/OmniSR_X2_DIV2K.safetensors -P /content/ComfyUI/models/upscale_models/
!wget -q --show-progress https://huggingface.co/Acly/Omni-SR/resolve/main/OmniSR_X3_DIV2K.safetensors -P /content/ComfyUI/models/upscale_models/
!wget -q --show-progress https://huggingface.co/Acly/Omni-SR/resolve/main/OmniSR_X4_DIV2K.safetensors -P /content/ComfyUI/models/upscale_models/

# Компоненты для Flux.2 Klein 4B
!wget -q --show-progress https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/vae/flux2-vae.safetensors -P /content/ComfyUI/models/vae/
!wget -q --show-progress https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors -P /content/ComfyUI/models/diffusion_models/

print("✅ Шаг 2: Модели скачаны.")

4x_NMKD-Superscale- 100%[===================>]  63.86M  79.7MB/s    in 0.8s    
MAT_Places512_G_fp1 100%[===================>] 119.48M  64.1MB/s    in 1.9s    
OmniSR_X2_DIV2K.saf 100%[===================>]   1.58M  3.60MB/s    in 0.4s    
OmniSR_X3_DIV2K.saf 100%[===================>]   1.60M  7.99MB/s    in 0.2s    
OmniSR_X4_DIV2K.saf 100%[===================>]   1.62M  6.84MB/s    in 0.2s    
qwen_3_4b.safetenso 100%[===================>]   7.49G   105MB/s    in 87s     
flux2-vae.safetenso 100%[===================>] 320.64M  36.8MB/s    in 5.4s    
✅ Шаг 2: Модели скачаны.


In [3]:
# Скачиваем и устанавливаем cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("✅ Шаг 3: Туннель установлен.")

Selecting previously unselected package cloudflared.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
✅ Шаг 3: Туннель установлен.


In [5]:
!grep -i "error" /content/comfy.log | head -n 30

In [4]:
import time

# Запускаем ComfyUI в фоне
!nohup python /content/ComfyUI/main.py --dont-print-server > /content/comfy.log 2>&1 &
print("Ждем загрузки ComfyUI (15 секунд)...")
time.sleep(15)
print("Запускаем туннель Cloudflare. Ссылка появится ниже...")

# Запускаем туннель (не останавливайте эту ячейку!)
!cloudflared tunnel --url http://127.0.0.1:8188

Ждем загрузки ComfyUI (15 секунд)...
Запускаем туннель Cloudflare. Ссылка появится ниже...
2026-11-19T12:04:02Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T12:04:02Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T12:04:06Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T12:04:06Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reach

In [6]:
!python /content/ComfyUI/main.py

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-11-19 12:09:39.724
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
Using Python 3.13.16

In [7]:
import time

# 1. Освобождаем порт 8188 (убиваем зависший старый процесс)
!fuser -k 8188/tcp

print("Старый сервер остановлен. Ждем 2 секунды...")
time.sleep(2)

# 2. Запускаем ComfyUI в фоне
!nohup python /content/ComfyUI/main.py --dont-print-server > /content/comfy.log 2>&1 &
print("Ждем загрузки ComfyUI (15 секунд)...")
time.sleep(15)

# 3. Запускаем туннель
print("Запускаем туннель Cloudflare. Ссылка появится ниже...")
!cloudflared tunnel --url http://127.0.0.1:8188

8188/tcp:             2865
Старый сервер остановлен. Ждем 2 секунды...
Ждем загрузки ComfyUI (15 секунд)...
Запускаем туннель Cloudflare. Ссылка появится ниже...
2026-11-19T12:11:55Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T12:11:55Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T12:11:59Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T12:11:59Z INF |  Your quick 

In [8]:
# Принудительно обновляем ключевые узлы до последней версии
!cd /content/ComfyUI/custom_nodes/comfyui-tooling-nodes && git pull
!cd /content/ComfyUI/custom_nodes/comfyui-inpaint-nodes && git pull

# На всякий случай переустанавливаем их зависимости
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui-tooling-nodes/requirements.txt
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui-inpaint-nodes/requirements.txt

print("✅ Узлы tooling и inpaint обновлены. Теперь перезапустите сервер.")


Already up to date.
Already up to date.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/ComfyUI/custom_nodes/comfyui-inpaint-nodes/requirements.txt'
✅ Узлы tooling и inpaint обновлены. Теперь перезапустите сервер.


In [9]:
import time

# Убиваем старые процессы, чтобы освободить порт
!fuser -k 8188/tcp

print("Ждем загрузки ComfyUI (15 секунд)...")
# Запускаем сервер в фоне
!nohup python /content/ComfyUI/main.py --dont-print-server > /content/comfy.log 2>&1 &
time.sleep(15)

print("Запускаем туннель Cloudflare. Ссылка появится ниже...")
# Запускаем туннель (не останавливайте эту ячейку!)
!cloudflared tunnel --url http://127.0.0.1:8188

8188/tcp:             5019
Ждем загрузки ComfyUI (15 секунд)...
Запускаем туннель Cloudflare. Ссылка появится ниже...
2026-11-19T12:15:27Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T12:15:27Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T12:15:31Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T12:15:31Z INF |  Your quick Tunnel has been created! Visit it at (it may

In [10]:
!echo "--- Diffusion Models ---"
!ls -la /content/ComfyUI/models/diffusion_models/
!echo "--- Text Encoders ---"
!ls -la /content/ComfyUI/models/text_encoders/
!echo "--- VAE ---"
!ls -la /content/ComfyUI/models/vae/

--- Diffusion Models ---
total 8
drwxr-xr-x  2 root root 4096 Nov 19 11:57 .
drwxr-xr-x 29 root root 4096 Nov 19 12:01 ..
-rw-r--r--  1 root root    0 Nov 19 11:57 put_diffusion_model_files_here
--- Text Encoders ---
total 7856440
drwxr-xr-x  2 root root       4096 Nov 19 12:01 .
drwxr-xr-x 29 root root       4096 Nov 19 12:01 ..
-rw-r--r--  1 root root          0 Nov 19 11:57 put_text_encoder_files_here
-rw-r--r--  1 root root 8044982048 Nov 19 12:02 qwen_3_4b.safetensors
--- VAE ---
total 328344
drwxr-xr-x  2 root root      4096 Nov 19 12:02 .
drwxr-xr-x 29 root root      4096 Nov 19 12:01 ..
-rw-r--r--  1 root root 336211292 Nov 19 12:03 flux2-vae.safetensors
-rw-r--r--  1 root root         0 Nov 19 11:57 put_vae_here


In [11]:
# Скачиваем недостающую модель Flux.2 Klein 4B (FP8)
!wget -q --show-progress https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors -P /content/ComfyUI/models/diffusion_models/

print("✅ Модель скачана. Проверяем папку:")
!ls -la /content/ComfyUI/models/diffusion_models/

✅ Модель скачана. Проверяем папку:
total 8
drwxr-xr-x  2 root root 4096 Nov 19 11:57 .
drwxr-xr-x 29 root root 4096 Nov 19 12:01 ..
-rw-r--r--  1 root root    0 Nov 19 11:57 put_diffusion_model_files_here


In [12]:
# 1. Проверяем, сколько места осталось на диске
print("--- Свободное место на диске ---")
!df -h /content

# 2. Скачиваем модель БЕЗ скрытия ошибок (-q убран)
print("\n--- Начинаем скачивание модели ---")
!wget --show-progress https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors -P /content/ComfyUI/models/diffusion_models/

# 3. Снова проверяем папку
print("\n--- Проверка папки diffusion_models ---")
!ls -lh /content/ComfyUI/models/diffusion_models/

--- Свободное место на диске ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   11G  102G  10% /

--- Начинаем скачивание модели ---
--2026-11-19 12:20:20--  https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors
Resolving huggingface.co (huggingface.co)... 99.86.101.64, 99.86.101.36, 99.86.101.39, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.64|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /Comfy-Org/Flux-2-Klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors [following]
--2026-11-19 12:20:20--  https://huggingface.co/Comfy-Org/Flux-2-Klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b-fp8.safetensors
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 404 Not Found
2026-11-19 12:20:20 ERROR 404: Not Found.


--- Проверка папки diffusion_models ---
tot